# 04 - Controlled IDRiD experiments

This notebook compares a small number of training choices. It changes **one major factor at a time** and uses only the training and validation images to choose a setup. The official test images are loaded only in the final cell.

**Important record check:** The README and older notebook text say the previous test accuracy was 81.4%. The current saved prediction file calculates to 77.5% accuracy and 0.877 AUC. Those records disagree. This notebook retrains a baseline under one fixed recipe and saves its own results so we can compare cleanly. It does not use test scores to choose an experiment.

The task remains binary: grade 0 = Normal; grades 1-4 = DR. This notebook is for research, not medical use.

## 1. Import the tools

In [3]:
from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("PyTorch:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch: 2.14.1+cpu
Device: CPU


## 2. Find the project folder and experiment output folder

In [5]:
ROOT_OVERRIDE = None  # If needed, set this to the project folder path.
if ROOT_OVERRIDE:
    ROOT = Path(ROOT_OVERRIDE).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    choices = [here, *here.parents, Path.home() / "Desktop" / "retinal-dr-classification"]
    ROOT = next((p for p in choices if (p / "data" / "IDRiD").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set ROOT_OVERRIDE to the project folder.")

RUN_DIR = ROOT / "reports" / "IDRiD_clean_run"
EXPERIMENT_DIR = ROOT / "reports" / "IDRiD_experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("New results will be saved in:", EXPERIMENT_DIR)

Project: C:\Users\PIXEL-PC\Desktop\retinal-dr-classification
New results will be saved in: C:\Users\PIXEL-PC\Desktop\retinal-dr-classification\reports\IDRiD_experiments


## 3. Load only training and validation manifests

We do not load the official test manifest in this section. The split files were made earlier and exact duplicate images were kept together.

In [7]:
train_data = pd.read_csv(RUN_DIR / "train_manifest.csv")
validation_data = pd.read_csv(RUN_DIR / "validation_manifest.csv")
print("Training images:", len(train_data))
print("Validation images:", len(validation_data))
counts = pd.DataFrame({
    "Training": train_data.label.value_counts().reindex([0, 1], fill_value=0),
    "Validation": validation_data.label.value_counts().reindex([0, 1], fill_value=0),
}).rename(index={0: "Normal", 1: "DR"})
display(counts)
print("The model target is label 0 = Normal and label 1 = DR.")

Training images: 329
Validation images: 83
        Training  Validation
label                       
Normal       106          27
DR           223          56
The model target is label 0 = Normal and label 1 = DR.


## 4. Decide what to compare before training

All runs use the same split, seed, model, batch size, optimizer, maximum epochs, and horizontal flip.

- **Baseline:** 224-pixel input and class-weighted loss.
- **Experiment A:** 300-pixel input; only input size changes. More pixels may preserve small retinal details, but training is slower.
- **Experiment B:** 224-pixel input and ordinary unweighted loss; only the imbalance strategy changes.

The baseline weights are calculated from the training labels only. No images are copied, removed, or synthetically created.

In [9]:
SEED = 20261008
EPOCHS = 8
BATCH_SIZE = 8
LEARNING_RATE = 0.0001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

experiments = [
    {"name": "baseline_224_weighted", "input_size": 224, "use_class_weights": True},
    {"name": "larger_input_300_weighted", "input_size": 300, "use_class_weights": True},
    {"name": "baseline_224_unweighted", "input_size": 224, "use_class_weights": False},
]
print(pd.DataFrame(experiments))

                        name  input_size  use_class_weights
0      baseline_224_weighted         224               True
1  larger_input_300_weighted         300               True
2    baseline_224_unweighted         224              False


## 5. Prepare images for one input size

The original image files stay unchanged. Resize and crop happen while an image is loaded for the model. Training images may be flipped left-to-right; validation images are not randomly changed.

In [11]:
class RetinalImages(Dataset):
    def __init__(self, table, input_size, training=False):
        self.table = table.reset_index(drop=True)
        self.input_size = input_size
        if training:
            self.transform = transforms.Compose([
                transforms.Resize(input_size + 32),
                transforms.CenterCrop(input_size),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(input_size + 32),
                transforms.CenterCrop(input_size),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ])

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        with Image.open(ROOT / row.relative_path) as image:
            image = image.convert("RGB")
        return self.transform(image), int(row.label), row.image_id

def make_loaders(input_size):
    train_loader = DataLoader(
        RetinalImages(train_data, input_size, training=True),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(
        RetinalImages(validation_data, input_size, training=False),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_loader, val_loader

print("Input loader is ready.")

Input loader is ready.


## 6. Define the simple score summary

Recall/sensitivity is the share of actual DR images found. Specificity is the share of normal images correctly recognized. AUC checks ranking across cutoffs; for the simple predictions below, the cutoff stays 0.5.

In [13]:
def measure_scores(labels, probabilities):
    predictions = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(labels, predictions),
        "sensitivity": recall_score(labels, predictions, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "precision": precision_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "roc_auc": roc_auc_score(labels, probabilities),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }

def predict(model, loader):
    model.eval()
    labels, probabilities, image_ids = [], [], []
    with torch.inference_mode():
        for images, batch_labels, batch_ids in loader:
            scores = model(images.to(DEVICE))
            probabilities.extend(torch.softmax(scores, dim=1)[:, 1].cpu().numpy())
            labels.extend(batch_labels.numpy())
            image_ids.extend(batch_ids)
    return np.asarray(labels), np.asarray(probabilities), image_ids

print("Score functions are ready.")

Score functions are ready.


## 7. Write the training step

For each setup, we train up to eight passes over the training images. After each pass, we check validation loss and keep the model with the lowest validation loss. The official test set is not involved.

In [15]:
def train_one_experiment(config):
    name = config["name"]
    input_size = config["input_size"]
    use_weights = config["use_class_weights"]

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train_loader, val_loader = make_loaders(input_size)
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    model = model.to(DEVICE)

    if use_weights:
        class_counts = train_data.label.value_counts().reindex([0, 1], fill_value=0)
        weights = len(train_data) / (2 * class_counts.to_numpy(dtype=np.float32))
        loss_function = nn.CrossEntropyLoss(
            weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
        print("Class weights Normal, DR:", weights.round(3))
    else:
        loss_function = nn.CrossEntropyLoss()
        print("No class weights are used in this run.")

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_loss = float("inf")
    best_epoch = 0
    history = []
    model_path = EXPERIMENT_DIR / f"{name}.pt"
    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_train_loss = 0.0
        seen = 0
        for images, labels, _ in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            scores = model(images)
            loss = loss_function(scores, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * len(labels)
            seen += len(labels)

        model.eval()
        total_val_loss = 0.0
        val_seen = 0
        with torch.inference_mode():
            for images, labels, _ in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                scores = model(images)
                loss = nn.functional.cross_entropy(scores, labels, reduction="sum")
                total_val_loss += loss.item()
                val_seen += len(labels)

        train_loss = total_train_loss / seen
        val_loss = total_val_loss / val_seen
        history.append({"epoch": epoch, "training_loss": train_loss, "validation_loss": val_loss})
        print(f"{name}: epoch {epoch}/{EPOCHS} - train loss {train_loss:.3f}, validation loss {val_loss:.3f}")
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch
            torch.save(model.state_dict(), model_path)

    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    y_val, p_val, val_ids = predict(model, val_loader)
    scores = measure_scores(y_val, p_val)
    result = {"name": name, "input_size": input_size, "class_weighted": use_weights,
              "best_epoch": best_epoch, "best_validation_loss": best_loss,
              "seconds": time.time() - start_time, **scores}

    pd.DataFrame(history).to_csv(EXPERIMENT_DIR / f"{name}_history.csv", index=False)
    pd.DataFrame({"image_id": val_ids, "true_label": y_val,
                  "dr_probability": p_val, "predicted_label": (p_val >= 0.5).astype(int)
                 }).to_csv(EXPERIMENT_DIR / f"{name}_validation_predictions.csv", index=False)
    (EXPERIMENT_DIR / f"{name}_config.json").write_text(
        json.dumps({**config, "seed": SEED, "epochs": EPOCHS,
                    "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
                    "selected_epoch": best_epoch, "selection_metric": "lowest validation loss"}, indent=2),
        encoding="utf-8")
    return result

print("Training function is ready. Each run saves its model, config, history, and validation predictions.")

Training function is ready. Each run saves its model, config, history, and validation predictions.


## 8. Run the baseline again

This is the reference run for this notebook: 224-pixel input with class-weighted loss.

In [17]:
baseline_result = train_one_experiment(experiments[0])
display(pd.Series(baseline_result, name="Baseline result").to_frame())

Class weights Normal, DR: [1.552 0.738]
baseline_224_weighted: epoch 1/8 - train loss 0.551, validation loss 0.410
baseline_224_weighted: epoch 2/8 - train loss 0.362, validation loss 0.268
baseline_224_weighted: epoch 3/8 - train loss 0.328, validation loss 0.298
baseline_224_weighted: epoch 4/8 - train loss 0.245, validation loss 0.230
baseline_224_weighted: epoch 5/8 - train loss 0.179, validation loss 0.187
baseline_224_weighted: epoch 6/8 - train loss 0.129, validation loss 0.169
baseline_224_weighted: epoch 7/8 - train loss 0.133, validation loss 0.220
baseline_224_weighted: epoch 8/8 - train loss 0.079, validation loss 0.307
                            Baseline result
name                  baseline_224_weighted
input_size                              224
class_weighted                         True
best_epoch                                6
best_validation_loss                 0.1694
seconds                          554.089225
accuracy                           0.831325
sensitiv

## 9. Experiment A - use a larger input image

Only input size changes from the baseline. We keep class weights and the other settings the same.

In [19]:
larger_result = train_one_experiment(experiments[1])
display(pd.Series(larger_result, name="Larger input result").to_frame())

Class weights Normal, DR: [1.552 0.738]
larger_input_300_weighted: epoch 1/8 - train loss 0.524, validation loss 0.377
larger_input_300_weighted: epoch 2/8 - train loss 0.349, validation loss 0.229
larger_input_300_weighted: epoch 3/8 - train loss 0.313, validation loss 0.242
larger_input_300_weighted: epoch 4/8 - train loss 0.230, validation loss 0.202
larger_input_300_weighted: epoch 5/8 - train loss 0.165, validation loss 0.187
larger_input_300_weighted: epoch 6/8 - train loss 0.113, validation loss 0.196
larger_input_300_weighted: epoch 7/8 - train loss 0.142, validation loss 0.170
larger_input_300_weighted: epoch 8/8 - train loss 0.080, validation loss 0.225
                            Larger input result
name                  larger_input_300_weighted
input_size                                  300
class_weighted                             True
best_epoch                                    7
best_validation_loss                   0.169535
seconds                             1168

## 10. Experiment B - remove class weights

Only the imbalance method changes from the baseline. This checks whether the weighting helps the validation results.

In [21]:
unweighted_result = train_one_experiment(experiments[2])
display(pd.Series(unweighted_result, name="Unweighted result").to_frame())

No class weights are used in this run.
baseline_224_unweighted: epoch 1/8 - train loss 0.548, validation loss 0.365
baseline_224_unweighted: epoch 2/8 - train loss 0.332, validation loss 0.266
baseline_224_unweighted: epoch 3/8 - train loss 0.298, validation loss 0.265
baseline_224_unweighted: epoch 4/8 - train loss 0.220, validation loss 0.218
baseline_224_unweighted: epoch 5/8 - train loss 0.170, validation loss 0.170
baseline_224_unweighted: epoch 6/8 - train loss 0.122, validation loss 0.166
baseline_224_unweighted: epoch 7/8 - train loss 0.148, validation loss 0.192
baseline_224_unweighted: epoch 8/8 - train loss 0.081, validation loss 0.270
                            Unweighted result
name                  baseline_224_unweighted
input_size                                224
class_weighted                          False
best_epoch                                  6
best_validation_loss                 0.165755
seconds                            934.569313
accuracy               

## 11. Compare validation results

Do not use the official test set to make this choice. The chosen run is the one with the highest validation AUC. If two are within 0.01 AUC, choose the one with higher DR sensitivity. These results are based on only 83 validation images, so small differences may be random.

In [23]:
results = pd.DataFrame([baseline_result, larger_result, unweighted_result])
results = results.sort_values(["roc_auc", "sensitivity"], ascending=False).reset_index(drop=True)
display(results[["name", "input_size", "class_weighted", "best_epoch",
                 "accuracy", "sensitivity", "specificity", "precision", "f1", "roc_auc",
                 "TN", "FP", "FN", "TP"]].style.format({
                     "accuracy":"{:.1%}", "sensitivity":"{:.1%}", "specificity":"{:.1%}",
                     "precision":"{:.1%}", "f1":"{:.1%}", "roc_auc":"{:.3f}"}))

best_name = results.iloc[0]["name"]
best_config = next(config for config in experiments if config["name"] == best_name)
print("Selected using validation only:", best_name)
results.to_csv(EXPERIMENT_DIR / "validation_experiment_comparison.csv", index=False)
(EXPERIMENT_DIR / "selected_experiment.json").write_text(
    json.dumps({"selected_name": best_name, "selection_rule": "highest validation ROC AUC; sensitivity breaks ties",
                "test_labels_used_for_selection": False}, indent=2), encoding="utf-8")

                        name  input_size  class_weighted  best_epoch  accuracy  sensitivity  specificity  precision        f1   roc_auc  TN  FP  FN  TP
0  larger_input_300_weighted         300            True           7  0.915663     0.946429     0.851852   0.929825  0.938053  0.982804  23   4   3  53
1    baseline_224_unweighted         224           False           6  0.843373     0.892857     0.740741   0.877193  0.884956  0.949735  20   7   6  50
2      baseline_224_weighted         224            True           6  0.831325     0.821429     0.851852   0.920000  0.867925  0.941799  23   4  10  46
Selected using validation only: larger_input_300_weighted


## 12. View the validation comparison

This chart helps compare the same measures across the three experiments. It is still validation data, not the final test.

In [25]:
results.set_index("name")[["accuracy", "sensitivity", "specificity", "roc_auc"]].plot(
    kind="bar", ylim=(0, 1), figsize=(10, 5), ylabel="Score", title="Validation results at 0.5 cutoff")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 13. One final score on the saved official test split

This cell comes after experiment selection. It loads the selected checkpoint and reports a test score once. A test score was already calculated earlier in this project, and the saved baseline records disagree (the README says 81.4%; current prediction CSV calculates 77.5%). Therefore this is a **re-check of a previously used test set**, not a new independent final evaluation. Do not use its results to revise the experiment choice. A genuinely independent final claim needs a new labeled test set.

In [27]:
test_data = pd.read_csv(RUN_DIR / "official_test_manifest.csv")
print("Test images:", len(test_data), "(loaded after validation selection)")
input_size = best_config["input_size"]
test_loader = DataLoader(RetinalImages(test_data, input_size, training=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(DEVICE)
model_path = EXPERIMENT_DIR / f"{best_name}.pt"
model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
y_test, p_test, test_ids = predict(model, test_loader)
test_result = measure_scores(y_test, p_test)
display(pd.Series(test_result, name="Selected model - test re-check").to_frame())
print("DR images missed:", test_result["FN"])
print("Normal images incorrectly flagged:", test_result["FP"])
pd.DataFrame({"image_id": test_ids, "true_label": y_test,
              "dr_probability": p_test, "predicted_label": (p_test >= 0.5).astype(int)
             }).to_csv(EXPERIMENT_DIR / "selected_model_test_predictions.csv", index=False)
summary = {"selected_experiment": best_name, "selection_rule": "validation ROC AUC, then sensitivity",
           "test_labels_used_for_selection": False, "test_is_independent": False,
           "test_results": {k: (float(v) if isinstance(v, (float, np.floating)) else int(v))
                            for k, v in test_result.items()}}
(EXPERIMENT_DIR / "experiment_run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

Test images: 102 (loaded after validation selection)
             Selected model - test re-check
accuracy                           0.794118
sensitivity                        0.750000
specificity                        0.882353
precision                          0.927273
f1                                 0.829268
roc_auc                            0.883651
TN                                30.000000
FP                                 4.000000
FN                                17.000000
TP                                51.000000
DR images missed: 17
Normal images incorrectly flagged: 4


## What to conclude

The validation comparison tells us whether these three training choices look promising on this split. A small validation set can give unstable rankings, so a tiny metric improvement is not proof of a better model. We keep the selected model and all comparison files under `reports/IDRiD_experiments/`. The final cell re-checks the already-used official test set only after selection; for a truly independent final evaluation, collect a new labeled test set from a different source or camera.